# Regulatory Simulation Sandbox — Dev Log

## Objetivo

Simula o impacto em cascata de uma mudança operacional hipotética associada
a um artigo — composição real de `regulatory_knowledge_graph.find_related()`
+ `regulatory_sandbox.compare_scenarios()`.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from core.regulatory_simulation.impact import simulate_regulatory_change
from shared.schemas import DataCategory, LegalBasis, SandboxScenario

scenario = SandboxScenario(
    name="Biometria sem revisão",
    data_categories=[DataCategory.SENSITIVE],
    legal_basis=LegalBasis.NOT_DETERMINED,
    context={"data_subtype": "biometric", "automated_decision": True, "human_review": False},
)
result = simulate_regulatory_change("20", {"human_review": True}, [scenario])
print(f"Artigos diretamente relacionados ao Art. 20º: {result.directly_affected_articles}")
print(f"Cenários avaliados: {result.scenarios_evaluated} | com mudança de score: {result.scenarios_with_score_change}")
c = result.comparisons[0]
print(f"Score antes: {c.scenario_a.trust_score.score} | depois: {c.scenario_b.trust_score.score} | delta: {c.score_delta}")
print()
print(result.summary)

Artigos diretamente relacionados ao Art. 20º: ['9º', '38º', '18º']
Cenários avaliados: 1 | com mudança de score: 1
Score antes: 5.0 | depois: 70.0 | delta: 65.0

Mudança hipotética no Art. 20º (context_override={'human_review': True}): 3 artigo(s) diretamente relacionado(s) no grafo regulatório (9º, 38º, 18º). 1 de 1 cenário(s) de teste tiveram o trust score alterado.


"E se o Art. 20 passasse a exigir revisão humana sempre?" — resposta
concreta: o cenário de biometria sem revisão sobe de score 5.0 (piso de
`DENY`) para 70.0, e o grafo real mostra que os Art. 9º, 18º e 38º também
seriam afetados (todos referenciam ou são referenciados pelo Art. 20º no
corpus real).

## Testes e Handoff

```
"C:/Users/Yuri_/.venvs/athenagov-ai/Scripts/python.exe" -m pytest core/regulatory_simulation/tests -v
```

7/7 testes passando.

## Handoff Summary da extração V3/V4 (8 módulos)

Com estes 8 módulos, a parte realmente codificável de V3/V4 está
implementada com o mesmo rigor do V1/V2 — 57 testes novos, total do projeto
**445 testes passando**. Os 4 itens restantes (Neuro-Symbolic Governance,
Cognitive Architecture Governance, AGI & Civilization Risk Governance, AI
Diplomacy) continuam documentados só como design em
`docs/architecture/v3-frontier-research.md`/`v4-systemic-civilizational.md`
— nenhuma implementação responsável era possível para eles sem
infraestrutura/pesquisa que o projeto não tem.